<div style="background-color: #f1f8e9; border-left: 6px solid #558b2f; padding: 15px; margin-bottom: 20px; border-radius: 4px;">
    <h1 style="color: #2e7d32; margin: 0; font-family: sans-serif;">Processing VHR Satellite Imagery for Weed Mapping</h1>
    <p style="margin: 5px 0 0 0; color: #37474f; font-size: 1.1em;">
        Automated pipeline to process multi-source Very High Resolution (VHR) satellite data and extract mean spectral metrics.
    </p>
</div>

In [6]:
# 1. Standard Library Imports
import os
import gc
import math
import json
from pathlib import Path
import xml.etree.ElementTree as ET

# 2. Third-Party Scientific & GIS Imports
import cv2
import openpyxl
import numpy as np
import pandas as pd
import seaborn as sns
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.transforms import Affine2D
from matplotlib_scalebar.scalebar import ScaleBar
from scipy import stats
from scipy.spatial import distance
import statsmodels.api as sm
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# 3. Raster Processing & Geospatial Data Imports
import fiona
import rasterio
from rasterio import features
from rasterio.mask import mask
from rasterio.features import geometry_mask
from rasterio.windows import from_bounds
from rasterio.plot import show, reshape_as_image
from rasterio.warp import calculate_default_transform, reproject, Resampling
from rasterstats import zonal_stats

# 4. Machine Learning & Image Processing (Scikit-Learn / Scikit-Image)
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, f1_score, 
    balanced_accuracy_score, recall_score
)
from skimage.feature import graycomatrix, graycoprops
from skimage.registration import phase_cross_correlation

# 5. Environment & System Setup
import nest_asyncio
nest_asyncio.apply()

# GIS Drivers Configuration
fiona.drvsupport.supported_drivers['KML'] = 'rw'


**Paths and directories**

In [3]:
#BASE
path_base = Path(os.getcwd()).parent
#RASTER
ortho_path = Path(path_base) / "01_img_data/drone/DJI Matrice 300 RTK/2024-06-13_MaízPoveda_RGB_20m_Orthomosaic.tif"
raster_pln = Path(path_base) / "01_img_data/satellite/PlanetScope/20240613_111156_44_24fc_3B_AnalyticMS_SR_8b_harmonized_clip.tif"
img_final_pln = Path(path_base) / "01_img_data/satellite/PlanetScope/20240613_PlanetScope_clipped.tif"
ms_wv2_crudo = Path(path_base) / "01_img_data/satellite/worldview-2/24JUN11104156-M3DS-050518203010_01_P001.TIF" 
pan_wv2_crudo = Path(path_base) / "01_img_data/satellite/worldview-2/24JUN11104156-P3DS-050518203010_01_P001.TIF"
pansharp_esri = pan_wv2 = Path(path_base) / "01_img_data/satellite/worldview-2/24JUN11104156-M3DS-050518203010_01_P001_PAN_esri.TIF"
#SHAPEFILES
shp_path = Path(path_base) / "06_QGIS/23-05-2024_QGis_MaízLaPoveda/2024-05-23_LaPoveda_Maiz-Sorgo_Polígonos.shp"
kml_aoi = Path(path_base) / "05_AOIs/AOI_LaPoveda.kml"
shp_path_EPSG32630 = Path(path_base) / "06_QGIS/23-05-2024_QGis_MaízLaPoveda/2024-05-23_LaPoveda_Maiz-Sorgo_Polígonos_EPSG32630.shp"
shp_pln = Path(path_base) / "06_QGIS/23-05-2024_QGis_MaízLaPoveda/2024-05-23_LaPoveda_Maiz-Sorgo_Patches_PlnScope_EPSG32630.shp"
shp_wv2 = Path(path_base) / "06_QGIS/23-05-2024_QGis_MaízLaPoveda/2024-05-23_LaPoveda_Maiz-Sorgo_Patches_wv-02_EsquinaII_EPSG32630.shp"
shp_bounds = Path(path_base) / "06_QGIS/23-05-2024_QGis_MaízLaPoveda/2024_bounds_Maiz-Sorgo_plot.shp"
shp_bounds_wv2 = Path(path_base) / "06_QGIS/23-05-2024_QGis_MaízLaPoveda/2024_bounds_Maiz-Sorgo_plot_wv2.shp"
#XML
pln_xml = Path(path_base) /"01_img_data/satellite/PlanetScope/20240613_111156_44_24fc_3B_AnalyticMS_8b_metadata_clip.xml"
ms_wv2_xml = Path(path_base) /"00_downloads/EUSI-Vantor/26EUSI-0826-01-02-03 - 106977/050518203010_01/050518203010_01_P001_MUL/24JUN11104156-M3DS-050518203010_01_P001.XML"
pan_wv2_xml = Path(path_base) /"00_downloads/EUSI-Vantor/26EUSI-0826-01-02-03 - 106977/050518203010_01/050518203010_01_P001_PAN/24JUN11104156-P3DS-050518203010_01_P001.XML"
#Bottom of Atmosphere imagery (BOA)
out_pln = Path(path_base) / "01_img_data/satellite/PlanetScope/20240613_111156_44_24fc_3B_AnalyticMS_SR_8b_harmonized_BOA.tif"
out_pan_wv2 = Path(path_base) / "01_img_data/satellite/worldview-2/24JUN11104156-P3DS-050518203010_01_P001_BOA.TIF"
out_ms_wv2 = Path(path_base) / "01_img_data/satellite/worldview-2/24JUN11104156-M3DS-050518203010_01_P001_BOA.TIF"
#PANSHARPENING
pansharp_BROV = Path(path_base) / "01_img_data/satellite/worldview-2/24JUN11104156-M3DS-050518203010_01_P001_BOA_PAN_simple_brovey.TIF"
pansharp_MEAN = Path(path_base) / "01_img_data/satellite/worldview-2/24JUN11104156-M3DS-050518203010_01_P001_BOA_PAN_simple_mean.TIF"
pansharp_ESRI = Path(path_base) / "01_img_data/satellite/worldview-2/24JUN11104156-M3DS-050518203010_01_P001_BOA_PAN_esri.TIF"
pansharp_GS = Path(path_base) / "01_img_data/satellite/worldview-2/24JUN11104156-M3DS-050518203010_01_P001_BOA_PAN_gram_schmidt.TIF"

A function to recursively search a directory for images using rglob and extract key metadata, including platform-sensor details, spatial/spectral resolution, CRS, and dimensions

In [4]:
def extract_raster_data(root_dir_path):
    root_path = Path(root_dir_path)
    extensions = ['*.tif', '*.tiff', '*.vrt']
    img_files = []
    for ext in extensions:
        img_files.extend(root_path.rglob(ext))
    if not img_files:
        print(f"No images in  directory: {root_path}")
        return pd.DataFrame()

    metadata_records = [] #save each image metadata

    for img_path in img_files:
        try:
            with rasterio.open(img_path) as src:
                parts = img_path.parts
                platform = "Desconocida"
                sensor = "Desconocido"
                if "01_img_data" in parts:
                    idx = parts.index("01_img_data")
                    if idx + 1 < len(parts):
                        platform = parts[idx + 1]
                    if idx + 2 < len(parts):
                        sensor = parts[idx + 2]
                
                # Extract band name from metadata
                band_names = []
                for i in range(src.count):
                    name = src.descriptions[i]
                    if name:
                        band_names.append(name)
                    else:
                        band_names.append(f"Band_{i+1}")
                band_names_str = ", ".join(band_names)
                
                #Imagery characteristics dictionary
                record = {
                    "File": img_path.name,
                    "Plataform": platform,
                    "Sensor": sensor,
                    "Spatial res (m)": round(src.res[0], 4),
                    "Spectral res (bands)": src.count,
                    "Band names": band_names_str,
                    "CRS": str(src.crs) if src.crs else "No definido",
                    "Columns": src.width,
                    "Rows": src.height,
                    "Geographic bounds": f"L:{round(src.bounds.left,2)}, B:{round(src.bounds.bottom,2)}, R:{round(src.bounds.right,2)}, T:{round(src.bounds.top,2)}"
                }
                metadata_records.append(record)
                
        except Exception as e:
            print(f"File processing error {img_path.name}: {e}")
    df = pd.DataFrame(metadata_records)
    print("                METADATA FROM AVAILABLE RASTER DATA")
    print("="*80)
    print(df.to_string(index=False))
    return df

df_img = extract_raster_data(Path(path_base) / "01_img_data")

                METADATA FROM AVAILABLE RASTER DATA
                                                             File Plataform              Sensor  Spatial res (m)  Spectral res (bands)                                                     Band names        CRS  Columns   Rows                                   Geographic bounds
   20240613_111156_44_24fc_3B_AnalyticMS_SR_8b_harmonized_BOA.tif satellite         PlanetScope           3.0000                     8 Band_1, Band_2, Band_3, Band_4, Band_5, Band_6, Band_7, Band_8 EPSG:32630     1778   1734    L:455775.0, B:4460595.0, R:461109.0, T:4465797.0
  20240613_111156_44_24fc_3B_AnalyticMS_SR_8b_harmonized_clip.tif satellite         PlanetScope           3.0000                     8  coastal_blue, blue, green_i, green, yellow, red, rededge, nir EPSG:32630     1778   1734    L:455775.0, B:4460595.0, R:461109.0, T:4465797.0
                                 20240613_PlanetScope_clipped.tif satellite         PlanetScope           3.0000     

# Exploratory imagery-data analysis:

## PlanetScope (Planet Labs):

 ### Data Characteristics: 
 Planet delivers imagery in a standard 16-bit Digital Number (DN) format to optimize storage efficiency. The data includes a reflectanceCoefficient, which serves as a direct scaling factor to convert DN into Top-of-Atmosphere (TOA) reflectance.

 ### Metadata & Processing: 
 The accompanying .xml metadata file indicates that the imagery is geometrically corrected and orthorectified, but has not undergone atmospheric correction. Additionally, the dataset has already been harmonized to Sentinel-2 radiometric characteristics by Planet's processing pipeline.

## WorldView-2 (EUSI-Vantor):

 ### Data Characteristics: 
 Acquired in BUNDLE format (Panchromatic + Multispectral). The imagery is delivered as raw Digital Numbers (DN) stored in a 16-bit format and is not pre-calibrated to reflectance.

 ### Metadata & Processing: 
 Radiometric calibration relies on the accompanying .IMD (Image Metadata) text file. This file contains unique, acquisition-specific calibration factors tailored to the exact date of capture, providing independent parameters for both the panchromatic (PAN) and multispectral (MS) sensors. These parameters are required to calculate at-sensor radiance and subsequent TOA reflectance.

In [5]:
def audit_raster_dataset(raster_path, sensor_name):
    print(f"RADIOMETRIC AUDIT: {sensor_name}")
    print("-" * 50)
    
    with rasterio.open(raster_path) as src:
        # Basic metadata extraction
        print(f"1. Data Type (dtype):      {src.dtypes[0]}")
        print(f"2. Defined NoData Value:   {src.nodata}")
        print(f"3. Dimensions (W x H):     {src.width} columns x {src.height} rows")
        print(f"4. Total Spectral Bands:   {src.count}")
        
        scale_factor = 10
        out_shape = (src.count, int(src.height / scale_factor), int(src.width / scale_factor))
        
        # Read first and last band
        b1 = src.read(1, out_shape=(out_shape[1], out_shape[2]))
        
        # Determine the 1-based index of the NIR band depending on the sensor
        nir_idx = 7 if src.count == 8 and 'WV2' in sensor_name else src.count
        b_nir = src.read(nir_idx, out_shape=(out_shape[1], out_shape[2]))
        
        # Mask out NoData values to prevent radiometric distortion in descriptive statistics
        nodata_val = src.nodata if src.nodata is not None else 0
        b1_valid = b1[b1 != nodata_val]
        bnir_valid = b_nir[b_nir != nodata_val]
        
        if len(b1_valid) > 0:
            print("\nRadiometric stats (First and Last Spectral Bands):")
            print(f"   Band 1 (Coastal/Blue): Min={b1_valid.min():.2f} | Max={b1_valid.max():.2f} | Mean={b1_valid.mean():.2f}")
            print(f"   NIR Band:              Min={bnir_valid.min():.2f} | Max={bnir_valid.max():.2f} | Mean={bnir_valid.mean():.2f}")
        else:
            print("\n⚠️ WARNING: No valid pixel observations detected in the subsampled array.")
            
        print("=" * 60 + "\n")

# EXECUTION: Update with your path variables
audit_raster_dataset(ms_wv2_crudo, "WorldView-2 (2.0m)")
audit_raster_dataset(raster_pln, "PlanetScope (3.0m)")

RADIOMETRIC AUDIT: WorldView-2 (2.0m)
--------------------------------------------------
1. Data Type (dtype):      uint16
2. Defined NoData Value:   None
3. Dimensions (W x H):     2483 columns x 2797 rows
4. Total Spectral Bands:   8

Radiometric stats (First and Last Spectral Bands):
   Band 1 (Coastal/Blue): Min=318.00 | Max=1274.00 | Mean=404.59
   NIR Band:              Min=13.00 | Max=2047.00 | Mean=665.14

RADIOMETRIC AUDIT: PlanetScope (3.0m)
--------------------------------------------------
1. Data Type (dtype):      uint16
2. Defined NoData Value:   0.0
3. Dimensions (W x H):     1778 columns x 1734 rows
4. Total Spectral Bands:   8

Radiometric stats (First and Last Spectral Bands):
   Band 1 (Coastal/Blue): Min=28.00 | Max=5019.00 | Mean=689.91
   NIR Band:              Min=144.00 | Max=7063.00 | Mean=2487.66



> **[!NOTE]**
> Los dos satélites deben estar en **Bottom of Atmosphere (BOA)**. 
> Nosotros obtenemos las imágenes en TOA de Planet y en números digitales de wv-2. Tenemos influencia de la atmósfera en los valores espectrales.

In [ ]:
print("INITIALIZING AUTOMATED MULTI-SENSOR IMAGE CORRECTION PIPELINE (WV2 & PLANET)")

def extract_calibration_factors(xml_path, sensor_name, is_multispectral=True):
    print(f"\n📖 Reading metadata ({sensor_name}): {os.path.basename(xml_path)}")
    tree = ET.parse(xml_path)
    root = tree.getroot()
    factors = {}
    print(f"\n🔍 VISUAL VALIDATION OF EXTRACTED CALIBRATION FACTORS ({sensor_name}):")
    print("-" * 95)
    
    if sensor_name.upper() == "WV2":
        # Solar Spectral Irradiance (Esun) constants according to Updike & Comp (2010)
        esun_wv2 = {1: 1758.22, 2: 1974.24, 3: 1845.99, 4: 1488.47,
                    5: 1038.06, 6: 1161.08, 7: 825.26, 8: 473.66}
        esun_pan = 1577.06
        
        # Extract astronomical and solar geometry parameters
        try:
            sun_elev = float(root.find('.//MEANSUNEL').text)
            dist_node = root.find('.//EARTHSUNDISTANCE')
            earth_sun_dist = float(dist_node.text) if dist_node is not None else 1.0
            print(f"Solar Elevation Angle: {sun_elev}° | Earth-Sun Distance: {earth_sun_dist} AU")
        except AttributeError:
            sun_elev, earth_sun_dist = 60.0, 1.0
            print("WARNING: Solar geometry parameters not found. Applying default values.")

        # Geometrical correction factor (incorporating earth-sun distance and solar elevation angle)
        geom_factor = (earth_sun_dist**2 * math.pi) / math.sin(math.radians(sun_elev))
        
        print("-" * 95)
        print(f"{'Band':<8} | {'AbsCalFactor':<15} | {'EffBandwidth':<15} | {'Esun':<8} | {'Final TOA Refl. Mult':<25}")
        print("-" * 95)
        
        if is_multispectral: # Multispectral
            band_mapping = {'BAND_C': 1, 'BAND_B': 2, 'BAND_G': 3, 'BAND_Y': 4,
                            'BAND_R': 5, 'BAND_RE': 6, 'BAND_N': 7, 'BAND_N2': 8}
            for label, band_num in band_mapping.items():
                node = root.find(f'.//{label}')
                if node is not None:
                    abs_cal = float(node.find('.//ABSCALFACTOR').text)
                    eff_bw = float(node.find('.//EFFECTIVEBANDWIDTH').text)
                    
                    spectral_radiance = abs_cal / eff_bw
                    final_multiplier = (spectral_radiance * geom_factor) / esun_wv2[band_num]
                    
                    factors[band_num] = {'multiplier': final_multiplier}
                    print(f"Band {band_num:<2} | {abs_cal:<15.6e} | {eff_bw:<15.6e} | {esun_wv2[band_num]:<8} | {final_multiplier:<25.6e}")
        else: # Panchromatic
            pan_node = root.find('.//BAND_P')
            if pan_node is None: pan_node = root
            abs_cal = float(pan_node.find('.//ABSCALFACTOR').text)
            eff_bw = float(pan_node.find('.//EFFECTIVEBANDWIDTH').text)
            
            spectral_radiance = abs_cal / eff_bw
            final_multiplier = (spectral_radiance * geom_factor) / esun_pan
            factors[1] = {'multiplier': final_multiplier}
            print(f"PAN Band | {abs_cal:<15.6e} | {eff_bw:<15.6e} | {esun_pan:<8} | {final_multiplier:<25.6e}")

    elif sensor_name.upper() == "PLANET":
        print(f"{'Band':<8} | {'ReflectanceCoefficient (Direct Multiplier)':<40}")
        print("-" * 95)
        for elem in root.iter():
            if 'bandSpecificMetadata' in elem.tag:
                band_num = refl_coeff = None
                for child in elem.iter():
                    if 'bandNumber' in child.tag: band_num = int(child.text)
                    elif 'reflectanceCoefficient' in child.tag: refl_coeff = float(child.text)
                
                if band_num is not None and refl_coeff is not None:
                    factors[band_num] = {'multiplier': refl_coeff}
                    print(f"Band {band_num:<2} | {refl_coeff:<40.6e}")
                    
    print("-" * 95)
    input("⚠️ WARNING: Verify the metadata values. If they are correct, press ENTER to proceed (or Ctrl+C to abort)...")
    
    return factors

def execute_automated_processing(tif_path, xml_path, out_path, sensor_name, is_multispectral=True):
    factors = extract_calibration_factors(xml_path, sensor_name, is_multispectral)
    print(f"\n Processing {sensor_name} | Dataset: {os.path.basename(tif_path)}")
    with rasterio.open(tif_path) as src:
        meta = src.meta.copy()
        meta.update(dtype=rasterio.float32, nodata=0.0)
        nodata_in = src.nodata if src.nodata is not None else 0
        
        with rasterio.open(out_path, 'w', **meta) as dst:
            for i in range(1, src.count + 1):
                if i in factors:
                    band_data = src.read(i)
                    multiplier = factors[i]['multiplier']
                    mask_valid = (band_data != nodata_in) & (band_data > 0)
                    
                    # A. TOA Reflectance Conversion
                    toa_reflectance = np.where(mask_valid, band_data.astype(np.float32) * multiplier, 0.0)
                    
                    # B. BOA Reflectance (DOS1 Atmospheric Correction)
                    if mask_valid.any():
                        dark_object = np.percentile(toa_reflectance[mask_valid], 0.01)
                        boa_reflectance = np.where(mask_valid, np.clip(toa_reflectance - dark_object, a_min=0.0001, a_max=None), 0.0)
                        dst.write(boa_reflectance.astype(np.float32), i)
                        print(f"  -> Band {i:02d} processed | Subtracted atmospheric path radiance: {dark_object:.4e} | OK")
                    else:
                        dst.write(toa_reflectance, i)
                else:
                    print(f"  -> WARNING: Band {i} omitted (No calibration factors found in metadata).")             
    print(f"Successfully saved to: {out_path}")

if __name__ == "__main__":
    # Change sensor_name to "PLANET" to process PlanetScope imagery
    execute_automated_processing(pan_wv2_crudo, pan_wv2_xml, out_pan_wv2, sensor_name="WV2", is_multispectral=False)

## Imagery and Shapefile Preprocessing

 ### CRS Alignment: 
    Reprojected the shapefile's Coordinate Reference System (CRS) from EPSG:25830 to EPSG:32630 to match the projection of the satellite imagery.

 ### Parallax Displacement: 
 A pronounced parallax error was observed in the WorldView-2 imagery, causing a southwestward shift of approximately 5 meters along both the X and Y axes.

 ### Spatial Adjustment (Translation): 
 To ensure geometric comparability between the datasets, two separate stand shapefiles were utilized. The polygons were spatially adjusted to represent the exact same ground area across both images using a translation vector anchored to a specific, identifiable parcel corner.

In [ ]:
# Points in the bottom-left corner measured in QGIS
x_II_drone = 458560.0316  
y_II_drone = 4462647.4689 
x_II_wv2 = 458553.989   
y_II_wv2 = 4462650.133  

def translate_and_save_boundaries(x_origin, y_origin, x_target, y_target, input_path, output_path):
    dx = x_target - x_origin
    dy = y_target - y_origin
    print(f" -> Calculated spatial translation vector: X={dx:.3f} m, Y={dy:.3f} m")
    boundaries = gpd.read_file(input_path)
    corrected_boundaries = boundaries.copy() 
    corrected_boundaries['geometry'] = corrected_boundaries.geometry.translate(xoff=dx, yoff=dy)
    corrected_boundaries.to_file(output_path)
    return corrected_boundaries

gdf_wv2 = translate_and_save_boundaries(
    x_origin = x_II_drone, 
    y_origin = y_II_drone,
    x_target = x_II_wv2,
    y_target = y_II_wv2,
    input_path = shp_bounds,
    output_path = shp_bounds_wv2
)

Iniciando proceso de desplazamiento...
 -> Vector de traslación calculado: X=-6.043 m, Y=2.664 m
 -> Cargando archivo original en memoria...
 -> Aplicando corrección geométrica...
 -> Guardando archivo corregido en disco:
    C:\Users\48755625H\Nextcloud\PhD ICA-CSIC\ESA_PP0106977\06_QGIS\23-05-2024_QGis_MaízLaPoveda\2024_bounds_Maiz-Sorgo_plot_wv2.shp
✅ Proceso completado con éxito.




### Pan-sharpening (image data fusion method)

- Four pansharpening algorithms were evaluated and compared to enhance the spatial resolution of WorldView-2 multispectral imagery from 2.0 m to 0.5 m: Modified Brovey, Simple Mean, ESRI method, and Gram-Schmidt (GS). The Gram-Schmidt method was selected for the final quantitative analysis because it simulates the panchromatic band response by averaging the multispectral bands, thereby maximizing the preservation of vegetation spectral signatures.
- Reference original python script for simple pan-sharpening methods:  https://github.com/ThomasWangWeiHong/Simple-Pansharpening-Algorithms 

In [ ]:
def pansharpen(m, pan, method='esri'):
    """ 
    Pan-sharpening for N-bands.
    Methods available: 'simple_brovey', 'simple_mean', 'esri', 'gram_schmidt'.
    """
    m_dir = os.path.dirname(m)
    m_name, m_ext = os.path.splitext(os.path.basename(m))
    psh = os.path.join(m_dir, f"{m_name}_PAN_{method}{m_ext}")
    # Read multispectral
    with rasterio.open(m) as f:
        metadata_ms = f.profile
        img_ms = np.transpose(f.read(), [1, 2, 0]) 
    # Read panchromatic
    with rasterio.open(pan) as g:
        metadata_pan = g.profile
        img_pan = g.read(1)
    ms_to_pan_ratio = metadata_ms['transform'][0] / metadata_pan['transform'][0]
    rescaled_ms = cv2.resize(img_ms, dsize=None, fx=ms_to_pan_ratio, fy=ms_to_pan_ratio, 
                             interpolation=cv2.INTER_CUBIC).astype(metadata_ms['dtype'])
    if img_pan.shape[0] < rescaled_ms.shape[0]:
        rescaled_ms = rescaled_ms[: img_pan.shape[0], :, :]
    else:
        img_pan = img_pan[: rescaled_ms.shape[0], :]
    if img_pan.shape[1] < rescaled_ms.shape[1]:
        rescaled_ms = rescaled_ms[:, : img_pan.shape[1], :]
    else:
        img_pan = img_pan[:, : rescaled_ms.shape[1]]
    del img_ms; gc.collect()
    # Total bands matrix empty
    img_psh = np.zeros(rescaled_ms.shape, dtype=metadata_pan['dtype'])
    metadata_pan['height'] = rescaled_ms.shape[0]
    metadata_pan['width'] = rescaled_ms.shape[1]
    
    # PAN-SHARPENING METHODS EXPLORED
    # Simple Brovey
    if method == 'simple_brovey':
        all_in = np.sum(rescaled_ms, axis=2).astype(np.float32)
        safe_all_in = np.where(all_in == 0, 1, all_in)
        ratio = img_pan / safe_all_in
        for band in range(rescaled_ms.shape[2]):
            img_psh[:, :, band] = np.multiply(rescaled_ms[:, :, band], ratio) 
    # Simple Mean
    elif method == 'simple_mean':
        for band in range(rescaled_ms.shape[2]):
            img_psh[:, :, band] = 0.5 * (rescaled_ms[:, :, band] + img_pan)
    # ESRI method
    elif method == 'esri':
        ADJ = img_pan.astype(np.float32) - rescaled_ms.mean(axis=2)
        for band in range(rescaled_ms.shape[2]):
            banda_calc = rescaled_ms[:, :, band] + ADJ
            banda_calc = np.clip(banda_calc, 0, 65535)
            img_psh[:, :, band] = banda_calc        
    # GRAM-SCHMIDT method
    elif method == 'gram_schmidt':
        # PAN simulated
        sim_pan = np.mean(rescaled_ms, axis=2).astype(np.float32)
        # Mask to ignore black bounds
        mask = sim_pan > 0
        if np.any(mask):
            # 2. Igualar el histograma de la PAN real a la simulada
            mean_sim = np.mean(sim_pan[mask])
            std_sim = np.std(sim_pan[mask])
            mean_pan = np.mean(img_pan[mask])
            std_pan = np.std(img_pan[mask])
            if std_pan == 0: std_pan = 1e-6 
            # PAN statistically adjusted
            pan_matched = (img_pan.astype(np.float32) - mean_pan) * (std_sim / std_pan) + mean_sim
            
            # Calculate spatial details inyection
            var_sim = std_sim ** 2
            if var_sim == 0: var_sim = 1e-6
            for band in range(rescaled_ms.shape[2]):
                band_data = rescaled_ms[:, :, band].astype(np.float32)
                mean_band = np.mean(band_data[mask])
                cov = np.mean((band_data[mask] - mean_band) * (sim_pan[mask] - mean_sim))
                # Specific band inyection
                weight = cov / var_sim
                # Final equation for Gram-Schmidt: MS_new = MS_original + Weight * (PAN_ajusted - PAN_simulated)
                banda_calc = band_data + weight * (pan_matched - sim_pan)
                banda_calc = np.clip(banda_calc, 0, 65535)
                img_psh[:, :, band] = banda_calc
        else:
            print("WARNING: Empty image in Gram-Schmidt.")
            img_psh = rescaled_ms
    else:
        print("Method not implemented.")
        return None
    del img_pan, rescaled_ms; gc.collect()
    metadata_pan['count'] = img_psh.shape[2]
    with rasterio.open(psh, 'w', **metadata_pan) as dst:
        dst.write(np.transpose(img_psh, [2, 0, 1]))
    print(f"Pan-sharpening guardado en: {psh}")
    return img_psh

img_simple_brovey = pansharpen(out_ms_wv2, out_pan_wv2, method='simple_brovey')
img_simple_mean = pansharpen(out_ms_wv2, out_pan_wv2, method='simple_mean')
img_esri = pansharpen(out_ms_wv2, out_pan_wv2, method='esri')
img_gram_schmidt = pansharpen(out_ms_wv2, out_pan_wv2, method='gram_schmidt')

'\n\ndef pansharpen(m, pan, method=\'esri\'):\n    """ \n    Pan-sharpening universal para N-bandas.\n    Métodos disponibles: \'simple_brovey\', \'simple_mean\', \'esri\', \'gram_schmidt\'.\n    """\n    # 1. Autogenerar ruta de salida\n    m_dir = os.path.dirname(m)\n    m_name, m_ext = os.path.splitext(os.path.basename(m))\n    psh = os.path.join(m_dir, f"{m_name}_PAN_{method}{m_ext}")\n\n    # 2. Leer multiespectral (Lee TODAS las bandas automáticamente)\n    with rasterio.open(m) as f:\n        metadata_ms = f.profile\n        img_ms = np.transpose(f.read(), [1, 2, 0]) # Carga todas las bandas\n\n    # 3. Leer pancromática\n    with rasterio.open(pan) as g:\n        metadata_pan = g.profile\n        img_pan = g.read(1)\n\n    # 4. Redimensionar Multiespectral al tamaño de la PAN\n    ms_to_pan_ratio = metadata_ms[\'transform\'][0] / metadata_pan[\'transform\'][0]\n    rescaled_ms = cv2.resize(img_ms, dsize=None, fx=ms_to_pan_ratio, fy=ms_to_pan_ratio, \n                           

### Spectral metrics extraction
- The *Sorghum halepense* patches shapefile (reprojected to EPSG:32630) was loaded. 
- The polygons were categorized by size (High, mMdium, Low, and non-sorghum), and the mean spectral value was computed for each individual feature.
- Extract spectral signatures considering all pixels intersecting the stand (all_touched=True) to prevent the loss of peripheral pixels.

In [ ]:
def extract_signatures(shp_path, raster_path, prefix, col_id='ID_patch', col_lbl='Patch_size'):
    print(f"[{prefix}] Extracting spectral signature (all_touched=True)...")
    gdf = gpd.read_file(shp_path)
    df = gdf[[col_id, col_lbl, 'Area_m2']].copy()
    with rasterio.open(raster_path) as src:
        nodata = src.nodata if src.nodata is not None else 0
        data = []
        for geom in gdf.geometry:
            try:
                out, _ = mask(src, [geom], crop=True, all_touched=True)
                m = np.ma.masked_equal(out, nodata)
                mean = m.mean(axis=(1, 2))
                data.append(mean.filled(np.nan) if np.ma.is_masked(mean) else mean)
            except ValueError:
                data.append(np.full(src.count, np.nan))
    data = np.array(data)
    for i in range(data.shape[1]):
        df[f'{prefix}_B{i+1}'] = data[:, i]
    return df

def calculate_indices(df, prefix, b_nir, b_red, b_rededge=None):
    df[f'{prefix}_NDVI'] = (df[f'{prefix}_B{b_nir}'] - df[f'{prefix}_B{b_red}']) / (df[f'{prefix}_B{b_nir}'] + df[f'{prefix}_B{b_red}'])
    if b_rededge:
        df[f'{prefix}_NDRE'] = (df[f'{prefix}_B{b_nir}'] - df[f'{prefix}_B{b_rededge}']) / (df[f'{prefix}_B{b_nir}'] + df[f'{prefix}_B{b_rededge}'])
    return df

COL_ID, COL_LBL = 'ID_patch', 'Patch_size'
# DATASET 1: PlanetScope (3m) 
df_planet = extract_signatures(shp_pln, out_pln, "PS_3m", COL_ID, COL_LBL)
df_planet = calculate_indices(df_planet, "PS_3m", b_nir=8, b_red=6, b_rededge=7) # Planet: B8=NIR, B6=Red, B7=RedEdge
df_planet_clean = df_planet.dropna().reset_index(drop=True)
# DATASET 2: WorldView-2 MS Original (2m)
df_wv2_2m = extract_signatures(shp_wv2, out_ms_wv2, "WV2_2m", COL_ID, COL_LBL)
df_wv2_2m = calculate_indices(df_wv2_2m, "WV2_2m", b_nir=7, b_red=5, b_rededge=6) # WV2: B7=NIR1, B5=Red, B6=RedEdge
df_wv2_2m_clean = df_wv2_2m.dropna().reset_index(drop=True)
# DATASET 3: WorldView-2 Pansharpened GS (0.5m) 
df_wv2_gs = extract_signatures(shp_wv2, pansharp_GS, "WV2_05m_GS", COL_ID, COL_LBL)
df_wv2_gs = calculate_indices(df_wv2_gs, "WV2_05m_GS", b_nir=7, b_red=5, b_rededge=6) 
df_wv2_gs_clean = df_wv2_gs.dropna().reset_index(drop=True)

print(" Extraction report by image:")
print("="*60)
print(f"1. PlanetScope (3m)               -> Patches included: {len(df_planet_clean)}")
print(f"2. WorldView-2 MS (2m)            -> Patches included: {len(df_wv2_2m_clean)}")
print(f"3. WorldView-2 Pansharp GS (0.5m) -> Patches included: {len(df_wv2_gs_clean)}")

output_path = Path(path_base) / "07_datasets"
output_path.mkdir(parents=True, exist_ok=True)
df_planet_clean.to_excel(output_path / "Dataset_1_Planet_3m.xlsx", index=False)
df_wv2_2m_clean.to_excel(output_path / "Dataset_2_WV2_MS_2m.xlsx", index=False)
df_wv2_gs_clean.to_excel(output_path / "Dataset_3_WV2_Pansharp_GS_05m.xlsx", index=False)